In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd
import argparse
import pathlib
import os

import numpy as np

import matplotlib.pyplot as plt

def set_fontsize(base_fontsize=15):
    fontsize = base_fontsize
    plt.rcParams.update({
        'font.size': fontsize,
        'axes.titlesize': fontsize * 1,
        'axes.labelsize': fontsize,
        'xtick.labelsize': fontsize * 0.8,
        'ytick.labelsize': fontsize * 0.8,
        'legend.fontsize': fontsize * 0.8,
        'font.family': "Arial"
    })

plt.style.use('default')

set_fontsize()

In [ ]:
town = "Bonn"
objective = "cases_and_conc"
cutoff_value = 0.05 # cutoff value for ensemble member selection (fraction of best models)

path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/multistart_results"

## Plot predictions for cases/concentration for different cutoff dates

In [ ]:
phase_cut_dates = ["2024-01-03", "2023-11-08", "2023-03-15"]
descriptions = ["8 weeks", "16 weeks", "50 weeks"]

#phase_cut_dates = ["2024-01-03", "2023-07-19", "2023-03-15"]
#descriptions = ["8 weeks", "32 weeks", "50 weeks"]

In [ ]:
# load pred
conc_data = {}
I_data = {}
for phase_cut_date in phase_cut_dates:
    conc_data[phase_cut_date] = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_log_concentration.npz")
    I_data[phase_cut_date] = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_I7_reported.npz")

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

import sys 

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from optimization import optimization_utils

# load data and base model
base_config = {
        # data selection settings
        "data_kwargs": {
            "town": "Bonn",
            "sampling_area": "North_South",
            "project": "both", # one of ESI_CorA, AMELAG
            "max_precipitation_subsetting": None, # one of None, dry, light_rain
            "substance_normalization": "flow", # one of None, PMMoV, flow
            "gene_target": "N1", # one of N1, N2
            "log_scale": True, # this only considers WW measurements, not case counts
        },

        "E0": 862.857, 
        "I0": 1294.286,
        "R0": 162092.04, # 92% of pop, based on https://www.rki.de/DE/Themen/Infektionskrankheiten/Infektionskrankheiten-A-Z/C/COVID-19-Pandemie/AK-Studien/Ergebnisse.html
        "phase_cut_date": phase_cut_date, # date to split data into two phases
        "dt": 0.2,
        "underreporting_model": "monotone_increasing"
}



In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, sharex=True, sharey=False, figsize=(13.4, 5.7), dpi=300, constrained_layout=True)

for i, phase_cut_date in enumerate(phase_cut_dates):
    hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_{objective}/hparams.json"

    with open(hparams_path) as f:
        hparams = json.load(f)

    base_config.update(hparams)
    base_config["phase_cut_date"] = phase_cut_date
    data = optimization_utils.two_phase_integrative_model_load_data(base_config)

    quantiles = {q: jnp.quantile(conc_data[phase_cut_date]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

    conc_median = quantiles[0.5]
    conc_low  = quantiles[0.025]
    conc_high = quantiles[0.975]

    axs[0,i].scatter(data["conc_dates_train"], data["conc_train"], label="Training/validation\n(concentration)", color="#3d85c6ff",alpha=0.7, s=15)
    axs[0,i].scatter(data["conc_dates_val"], data["conc_val"], label=None, color="#3d85c6ff",alpha=0.7, s=15)
    axs[0,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    valid_idx = (data["t_all_idx"]*base_config["dt"] >= int(hparams.get("T_max")))
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.15, label="95% CI")
    conc_low  = quantiles[0.05]
    conc_high = quantiles[0.95]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.3, label="90% CI")
    conc_low  = quantiles[0.25]
    conc_high = quantiles[0.75]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[0,i].plot(data["dates_all"][valid_idx], conc_median, label="Median", color="#8B1000")
    axs[0,i].set_title(f"{descriptions[i]}")


    quantiles_I = {q: jnp.quantile(I_data[phase_cut_date]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
    I7_median = quantiles_I[0.5]
    I7_low  = quantiles_I[0.025]
    I7_high = quantiles_I[0.975]

    axs[1,i].scatter(data["I_dates_train"], data["I_train"], label="Training/validation\n(reported cases)", color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["I_dates_val"], data["I_val"], label=None, color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["obs_dates_phase_2"], data["I_test"], label="Test", color="#595959", alpha=0.7, s=15)
    axs[1,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.15, label="95% CI")
    I7_low  = quantiles_I[0.05]
    I7_high = quantiles_I[0.95]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.3, label="90% CI")
    I7_low  = quantiles_I[0.25]
    I7_high = quantiles_I[0.75]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[1,i].plot(data["dates_all"][1:], I7_median, label="Median", color="#8B1000")


axs[0,0].set_ylabel(
    "Flow normalised\nconcentration\n[log(GU/(ld))]")
axs[1,0].set_ylabel("7-day moving \nsum of reported\ncases [#]")

axs[0,1].set_yticklabels([])
axs[0,2].set_yticklabels([])
axs[1,1].set_yticklabels([])
axs[1,2].set_yticklabels([])

axs[0,i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
axs[0,i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

plt.tight_layout()
plt.savefig(f"Bonn_timeframe_comparisons.png", dpi=300, bbox_inches='tight')

In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 5), dpi=300)

handles, labels = [], []
for ax in axs.ravel():
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll and ll not in labels:
            labels.append(ll)
            handles.append(hh)
order = [0, 6, 7, 1, 5, 4, 3, 2]
labels = [labels[i] for i in order]
handles = [handles[i] for i in order]

ax2.axis('off')
fig2.legend(handles, labels, frameon=False)

fig2.savefig("Bonn_timeframe_comparisons_legend.png", dpi=300)

In [ ]:
phase_cut_dates = ["2024-01-31", "2023-07-19"]
descriptions = ["4 weeks", "32 weeks"]

# load pred
conc_data = {}
I_data = {}
for phase_cut_date in phase_cut_dates:
    conc_data[phase_cut_date] = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_log_concentration.npz")
    I_data[phase_cut_date] = np.load(f"{path}/{phase_cut_date}_{objective}/visualizations_{cutoff_value}/ensemble_predictions_I7_reported.npz")

fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=False, figsize=(9.4, 5.7), dpi=300, constrained_layout=True)

for i, phase_cut_date in enumerate(phase_cut_dates):
    hparams_path = f"/home/iru-mls/marvin/ww_bonn_jax/{town}/optuna_best_{phase_cut_date}_{objective}/hparams.json"

    with open(hparams_path) as f:
        hparams = json.load(f)

    base_config.update(hparams)
    base_config["phase_cut_date"] = phase_cut_date
    data = optimization_utils.two_phase_integrative_model_load_data(base_config)

    quantiles = {q: jnp.quantile(conc_data[phase_cut_date]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}

    conc_median = quantiles[0.5]
    conc_low  = quantiles[0.025]
    conc_high = quantiles[0.975]

    axs[0,i].scatter(data["conc_dates_train"], data["conc_train"], label="Training/validation\n(concentration)", color="#3d85c6ff",alpha=0.7, s=15)
    axs[0,i].scatter(data["conc_dates_val"], data["conc_val"], label=None, color="#3d85c6ff",alpha=0.7, s=15)
    axs[0,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    valid_idx = (data["t_all_idx"]*base_config["dt"] >= int(hparams.get("T_max")))
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.15, label="95% CI")
    conc_low  = quantiles[0.05]
    conc_high = quantiles[0.95]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.3, label="90% CI")
    conc_low  = quantiles[0.25]
    conc_high = quantiles[0.75]
    axs[0,i].fill_between(data["dates_all"][valid_idx], conc_low, conc_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[0,i].plot(data["dates_all"][valid_idx], conc_median, label="Median", color="#8B1000")
    axs[0,i].set_title(f"{descriptions[i]}")


    quantiles_I = {q: jnp.quantile(I_data[phase_cut_date]["all"], q, axis=0) for q in [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975]}
    I7_median = quantiles_I[0.5]
    I7_low  = quantiles_I[0.025]
    I7_high = quantiles_I[0.975]

    axs[1,i].scatter(data["I_dates_train"], data["I_train"], label="Training/validation\n(reported cases)", color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["I_dates_val"], data["I_val"], label=None, color="goldenrod", alpha=0.7, s=15)
    axs[1,i].scatter(data["obs_dates_phase_2"], data["I_test"], label="Test", color="#595959", alpha=0.7, s=15)
    axs[1,i].axvline(pd.to_datetime(phase_cut_date), color="#595959", linestyle='--', label="Phase split")

    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.15, label="95% CI")
    I7_low  = quantiles_I[0.05]
    I7_high = quantiles_I[0.95]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.3, label="90% CI")
    I7_low  = quantiles_I[0.25]
    I7_high = quantiles_I[0.75]
    axs[1,i].fill_between(data["dates_all"][1:], I7_low, I7_high, color="#8B0000", alpha=0.45, label="50% CI")
    axs[1,i].plot(data["dates_all"][1:], I7_median, label="Median", color="#8B1000")



axs[0,0].set_ylabel(
    "Flow normalised\nconcentration\n[log(GU/(ld))]"
)
axs[1,0].set_ylabel("7-day moving \nsum of reported\ncases [#]")

axs[0,1].set_yticklabels([])
axs[1,1].set_yticklabels([])

axs[0,i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 5, 9)))
axs[0,i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

plt.tight_layout()
plt.savefig(f"Bonn_timeframe_comparisons_2.png", dpi=300, bbox_inches='tight')